In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard component imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
JupyterDash.infer_jupyter_proxy_config()

# OS and data handling
import os
import pandas as pd

# Import the CRUD Python module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "goodpassword"

# Connect to MongoDB using the CRUD class
db = AnimalShelter(username, password)

# Load all records initially (empty query = retrieve all)
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB '_id' column (causes issues in DataTable)
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load Grazioso Salvare logo and make it clickable (per specs)
logo_filename = 'Grazioso Salvare Logo.png'
encoded_logo = ''

if os.path.exists(logo_filename):
    encoded_logo = base64.b64encode(open(logo_filename, 'rb').read()).decode()

app.layout = html.Div([
    # Logo with hyperlink to SNHU (required branding)
    html.A(
        html.Img(src='data:image/png;base64,{}'.format(encoded_logo), 
                 style={'height': '80px'}),
        href='https://www.snhu.edu',
        target='_blank'
    ),
    
    # Unique identifier
    html.Center(html.B(html.H1('Nathan Vanderpool -- CS-340 Project Two Dashboard', style={'color': 'blue'}))),
    html.Center(html.B(html.H1('Grazioso Salvare Dashboard'))),
    html.Hr(),
    
    # Interactive Filter Options (RadioItems)
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin': '10px'}
        )
    ]),
    html.Hr(),
    
    # Interactive Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} 
                 for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        page_action='native',
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto', 'minWidth': '100%'},
        style_cell={'textAlign': 'left', 'minWidth': '100px', 
                    'whiteSpace': 'normal', 'height': 'auto'},
    ),
    html.Br(),
    html.Hr(),
    
    # Charts side-by-side
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),   # Pie chart
        html.Div(id='map-id', className='col s12 m6')      # Geolocation map
    ])
])

#############################################
# Callbacks / Controller Logic
#############################################

# Update data table based on filter selection
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)

def update_dashboard(filter_type):
    # Applies MongoDB queries based on Rescue Type from specifications table.
    if filter_type == 'Reset' or filter_type is None:
        query = {}
    
    elif filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    
    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", 
                              "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    
    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", 
                              "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    
    else:
        query = {}
    
    # Query database via CRUD module
    filtered_df = pd.DataFrame.from_records(db.read(query))
    
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)
    
    return filtered_df.to_dict('records')

# Update pie chart (improved version)
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)

def update_graphs(viewData):
    # Dynamically updates breed distribution pie chart. Groups rare breeds into 'Other' for cleaner full-dataset view.
    if viewData is None or len(viewData) == 0:
        return [html.P("No data available for chart")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    if 'breed' not in dff.columns or len(dff) == 0:
        return [html.P("No breed data")]
    
    # Group rare breeds into "Other" for better readability
    breed_counts = dff['breed'].value_counts()
    top_breeds = breed_counts.head(8)  # Show top 8 breeds
    other_count = breed_counts.iloc[8:].sum() if len(breed_counts) > 8 else 0
    
    if other_count > 0:
        top_breeds = top_breeds.copy()  # avoid modifying original
        top_breeds['Other'] = other_count
    
    fig = px.pie(
        names=top_breeds.index, 
        values=top_breeds.values, 
        title='Breed Distribution in Filtered Results'
    )
    fig.update_traces(textinfo='percent+label')
    
    return [dcc.Graph(figure=fig)]

# Highlight selected columns in table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)

def update_styles(selected_columns):
    # Highlights selected columns for better UX.
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]

# Update geolocation map for selected row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    # Updates map marker for the selected animal row.
    if viewData is None:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    row = index[0] if index and len(index) > 0 else 0
    
    # Safe extraction of location/name/breed
    try:
        lat = float(dff.iloc[row, 13]) if len(dff.columns) > 13 else 30.75
        lon = float(dff.iloc[row, 14]) if len(dff.columns) > 14 else -97.48
        breed = str(dff.iloc[row, 4]) if len(dff.columns) > 4 else "Unknown"
        name = str(dff.iloc[row, 9]) if len(dff.columns) > 9 else "Unknown"
    
    except:
        lat, lon = 30.75, -97.48
        breed, name = "Unknown", "Unknown"
    
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]

# Run the dashboard
if __name__ == '__main__':
    app.run_server(mode='jupyterlab', port=8052, debug=False) 

Connected to MongoDB successfully.


 * Running on http://127.0.0.1:8052/ (Press CTRL+C to quit)
127.0.0.1 - - [21/Jun/2026 21:48:35] "GET /_alive_9d24419b-9c96-4873-b109-34d6ef060be7 HTTP/1.1" 200 -
127.0.0.1 - - [21/Jun/2026 21:48:35] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/Jun/2026 21:48:35] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [21/Jun/2026 21:48:36] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [21/Jun/2026 21:48:36] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Jun/2026 21:48:36] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Jun/2026 21:48:36] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [21/Jun/2026 21:48:36] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Jun/2026 21:48:36] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [21/Jun/2026 21:48:37] "POST /_dash-update-component HTTP/1.1" 200 -
